# LNPHub In Vitro Expression 3-Class Subset

This notebook creates a study-aware in vitro expression subset from the full `lnphub_master_v2.0.0.csv` file. It keeps studies with more than 500 unique formulations, harmonizes delivery into three ordered classes, saves the subset, and prints/plots per-study class statistics.


## 0. Clone Private Repository

This clones or updates the private `Maburidi/lnphub-submission` repository from the `dev` branch.


In [ ]:
# Cell 1 - Clone private LNPHub repo from dev branch
from pathlib import Path
from getpass import getpass
import os
import subprocess

REPO_URL = "https://github.com/Maburidi/lnphub-submission.git"
BRANCH = "dev"
PROJECT_DIR = Path("/content/lnphub-submission")


def get_github_token():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata
        token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        token = ""
    return token or getpass("GitHub token for the private repo: ").strip()


def run(cmd, token=None):
    printable = " ".join(map(str, cmd))
    if token:
        printable = printable.replace(token, "***TOKEN***")
    print("$", printable)
    res = subprocess.run(cmd, text=True, capture_output=True)
    out = (res.stdout + res.stderr).replace(token or "", "***TOKEN***")
    if out.strip():
        print(out[-4000:])
    if res.returncode != 0:
        raise RuntimeError(f"Command failed with code {res.returncode}")
    return res


token = get_github_token()
auth_repo_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

if PROJECT_DIR.exists():
    print(f"Repo already exists: {PROJECT_DIR}")
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", auth_repo_url], token=token)
    run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "pull", "origin", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], token=token)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", auth_repo_url, str(PROJECT_DIR)], token=token)
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], token=token)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())
run(["git", "-C", str(PROJECT_DIR), "branch", "--show-current"])


## 1. Load Full LNPHub Data


In [ ]:
# Cell 2 - Imports and paths
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

PROJECT_DIR = Path("/content/lnphub-submission")
DATA_DIR = PROJECT_DIR / "data_subsets"
OUT_DIR = PROJECT_DIR / "data_subsets" / "in_vitro_expression_3class"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = [
    DATA_DIR / "lnphub_master_v2.0.0.csv",
    PROJECT_DIR / "lnphub_master_v2.0.0.csv",
    Path("/content/lnphub_master_v2.0.0.csv"),
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find lnphub_master_v2.0.0.csv in repo or /content.")

print("Using data file:", DATA_PATH)
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Full LNPHub shape:", df.shape)
display(df.head(3))


In [ ]:
# Cell 3 - Check relevant study/metric counts
required = ["study_id", "experimental_setting", "delivery_metric", "delivery", "lnp_id"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work = df.copy()
work["experimental_setting"] = work["experimental_setting"].astype(str).str.strip()
work["delivery_metric"] = work["delivery_metric"].astype(str).str.strip()
work["study_id"] = work["study_id"].astype(str).str.strip()
work["delivery"] = pd.to_numeric(work["delivery"], errors="coerce")

print("Rows by experimental setting:")
display(work["experimental_setting"].value_counts(dropna=False).reset_index(name="rows").rename(columns={"index": "experimental_setting"}))

print("")
print("Rows by expression-like delivery metric:")
display(
    work[work["delivery_metric"].str.contains("expression", case=False, na=False)]["delivery_metric"]
    .value_counts(dropna=False)
    .reset_index(name="rows")
    .rename(columns={"index": "delivery_metric"})
)


## 2. Select Eligible In Vitro Expression Studies


In [ ]:
# Cell 4 - Filter in vitro expression/expression-discretized studies with >500 formulations
MIN_FORMULATIONS = 500
EXPRESSION_METRICS = {"expression", "expression_discretized", "expression_descritized"}

candidate = work[
    (work["experimental_setting"] == "in_vitro")
    & (work["delivery_metric"].isin(EXPRESSION_METRICS))
    & (work["delivery"].notna())
].copy()

study_summary = (
    candidate.groupby("study_id", dropna=False)
    .agg(
        rows=("study_id", "size"),
        formulations=("lnp_id", "nunique"),
        metrics=("delivery_metric", lambda s: ", ".join(sorted(s.dropna().unique()))),
        delivery_min=("delivery", "min"),
        delivery_median=("delivery", "median"),
        delivery_max=("delivery", "max"),
        delivery_unique_values=("delivery", "nunique"),
    )
    .reset_index()
    .sort_values("formulations", ascending=False)
)

eligible_studies = study_summary.loc[study_summary["formulations"] > MIN_FORMULATIONS, "study_id"].tolist()
subset = candidate[candidate["study_id"].isin(eligible_studies)].copy()

print(f"Candidate in vitro expression rows: {len(candidate):,}")
print(f"Eligible studies with >{MIN_FORMULATIONS:,} unique formulations: {len(eligible_studies):,}")
print(f"Eligible subset rows before 3-class harmonization: {len(subset):,}")

print("")
print("Eligible study summary:")
display(study_summary[study_summary["study_id"].isin(eligible_studies)])


## 3. Harmonize Delivery Into Three Classes


In [ ]:
# Cell 5 - Create harmonized 3-class delivery label: 0=low, 1=medium, 2=high
CLASS_LABELS = {0: "low", 1: "medium", 2: "high"}


def tertile_classes(values):
    values = pd.to_numeric(values, errors="coerce")
    ranks = values.rank(method="first")
    return pd.qcut(ranks, q=3, labels=[0, 1, 2]).astype(int)


def collapse_discrete_to_three(values):
    values = pd.to_numeric(values, errors="coerce")
    counts = values.value_counts(dropna=True)
    groups = [[float(v)] for v in sorted(counts.index)]

    while len(groups) > 3:
        centers = [np.mean(g) for g in groups]
        distances = [centers[i + 1] - centers[i] for i in range(len(groups) - 1)]
        min_dist = min(distances)
        tied = [i for i, d in enumerate(distances) if np.isclose(d, min_dist)]

        def merged_count(i):
            return sum(int(counts.loc[v]) for v in groups[i] + groups[i + 1])

        merge_i = min(tied, key=merged_count)
        groups[merge_i] = groups[merge_i] + groups[merge_i + 1]
        del groups[merge_i + 1]

    mapping = {}
    for cls, group in enumerate(groups):
        for v in group:
            mapping[v] = cls
    return values.astype(float).map(mapping).astype("Int64"), mapping


class_parts = []
mapping_rows = []

for (study_id, metric), sdf in subset.groupby(["study_id", "delivery_metric"], sort=False):
    sdf = sdf.copy()
    n_unique = sdf["delivery"].nunique(dropna=True)

    if metric in {"expression_discretized", "expression_descritized"}:
        if n_unique == 3:
            ordered = sorted(pd.to_numeric(sdf["delivery"], errors="coerce").dropna().unique())
            mapping = {float(v): i for i, v in enumerate(ordered)}
            sdf["delivery_class_3"] = pd.to_numeric(sdf["delivery"], errors="coerce").astype(float).map(mapping).astype("Int64")
            rule = "already_3_discrete_levels_preserved_order"
        elif n_unique in {4, 5}:
            sdf["delivery_class_3"], mapping = collapse_discrete_to_three(sdf["delivery"])
            rule = f"collapsed_{n_unique}_discrete_levels_to_3_by_closest_adjacent_values"
        else:
            sdf["delivery_class_3"] = tertile_classes(sdf["delivery"])
            mapping = {}
            rule = f"unexpected_{n_unique}_discrete_levels_used_tertiles"
    else:
        sdf["delivery_class_3"] = tertile_classes(sdf["delivery"])
        mapping = {}
        rule = "continuous_expression_tertiles_within_study_metric"

    sdf["delivery_class_label"] = sdf["delivery_class_3"].map(CLASS_LABELS)
    class_parts.append(sdf)
    mapping_rows.append({
        "study_id": study_id,
        "delivery_metric": metric,
        "n_rows": len(sdf),
        "n_formulations": sdf["lnp_id"].nunique(),
        "n_unique_delivery_values": int(n_unique),
        "harmonization_rule": rule,
        "discrete_value_to_class_mapping": mapping,
    })

subset_3class = pd.concat(class_parts, ignore_index=True)
mapping_table = pd.DataFrame(mapping_rows)

print("Harmonization rules by study/metric:")
display(mapping_table)

print("")
print("Class sanity check:")
display(
    subset_3class.groupby(["study_id", "delivery_metric", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
    .sort_values(["study_id", "delivery_metric", "delivery_class_3"])
)


## 4. Save Subset and Source Tables


In [ ]:
# Cell 6 - Save harmonized subset and summary tables
subset_path = OUT_DIR / "lnphub_in_vitro_expression_gt500formulations_3class.csv"
study_summary_path = OUT_DIR / "study_eligibility_summary.csv"
mapping_path = OUT_DIR / "delivery_3class_harmonization_rules.csv"
class_counts_path = OUT_DIR / "study_class_counts.csv"

class_counts = (
    subset_3class.groupby(["study_id", "delivery_class_3", "delivery_class_label"])
    .agg(rows=("study_id", "size"), formulations=("lnp_id", "nunique"))
    .reset_index()
    .sort_values(["study_id", "delivery_class_3"])
)

subset_3class.to_csv(subset_path, index=False)
study_summary.to_csv(study_summary_path, index=False)
mapping_table.to_csv(mapping_path, index=False)
class_counts.to_csv(class_counts_path, index=False)

print("Saved subset:", subset_path)
print("Saved study summary:", study_summary_path)
print("Saved harmonization rules:", mapping_path)
print("Saved class counts:", class_counts_path)
print("")
print("Final subset shape:", subset_3class.shape)
print("Studies:", subset_3class["study_id"].nunique())
print("Unique formulations:", subset_3class["lnp_id"].nunique())

display(class_counts)


## 5. Plot Per-Study Class Balance


In [ ]:
# Cell 7 - Bar plots of low / medium / high class counts per study
plot_df = class_counts.copy()
plot_df["class"] = pd.Categorical(plot_df["delivery_class_label"], categories=["low", "medium", "high"], ordered=True)

order = (
    subset_3class.groupby("study_id")["lnp_id"]
    .nunique()
    .sort_values(ascending=False)
    .index
    .tolist()
)

palette = {"low": "#4E79A7", "medium": "#A0A7B4", "high": "#D55E00"}

fig, ax = plt.subplots(figsize=(9.2, max(4.2, 0.34 * len(order))))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

sns.barplot(
    data=plot_df,
    y="study_id",
    x="rows",
    hue="class",
    order=order,
    hue_order=["low", "medium", "high"],
    palette=palette,
    ax=ax,
    edgecolor="white",
    linewidth=0.5,
)

ax.set_xlabel("Rows")
ax.set_ylabel("")
ax.grid(axis="x", color="#E5E7EB", linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(title="3-class delivery", frameon=False, loc="lower right")
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig_path = FIG_DIR / "in_vitro_expression_3class_counts_by_study.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()


In [ ]:
# Cell 8 - Faceted per-study distribution view
plot_df = subset_3class.copy()
plot_df["delivery_class_label"] = pd.Categorical(
    plot_df["delivery_class_label"], categories=["low", "medium", "high"], ordered=True
)

n_studies = plot_df["study_id"].nunique()
ncols = 3
nrows = int(np.ceil(n_studies / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(9.2, max(3.2, 2.05 * nrows)), squeeze=False)
fig.patch.set_facecolor("white")

for ax in axes.flat:
    ax.axis("off")

for ax, study_id in zip(axes.flat, order):
    sdf = plot_df[plot_df["study_id"] == study_id]
    counts = sdf["delivery_class_label"].value_counts().reindex(["low", "medium", "high"], fill_value=0)
    ax.axis("on")
    ax.bar(counts.index.astype(str), counts.values, color=[palette[k] for k in counts.index], width=0.72)
    ax.set_title(study_id, fontsize=9, loc="left", pad=3)
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for x, y in enumerate(counts.values):
        ax.text(x, y, f"{int(y):,}", ha="center", va="bottom", fontsize=7)

fig.tight_layout(h_pad=1.0, w_pad=1.0)
fig_path = FIG_DIR / "in_vitro_expression_3class_counts_faceted.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()
